In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms.functional import resize
from torchvision.transforms import InterpolationMode
import random
from sklearn.model_selection import train_test_split

In [2]:
os.chdir('/home/ntdung/Medical')

In [3]:
excel_path = 'data/participants.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66.0,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69.0,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23.0,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54.0,control,f,sub-BrainAge023212,RocklandSample


In [4]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4242
Samples with decimal age values: 706


In [5]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54,control,f,sub-BrainAge023212,RocklandSample


In [6]:
def load_middle_slices(nii_path):
    img = nib.load(nii_path).get_fdata()

    x, y, z = img.shape
    axial_slice = img[:, :, z // 2]
    sagittal_slice = img[x // 2, :, :]
    coronal_slice = img[:, y // 2, :]

    # Normalize
    axial_slice = (axial_slice - axial_slice.min()) / (axial_slice.max() - axial_slice.min())
    sagittal_slice = (sagittal_slice - sagittal_slice.min()) / (sagittal_slice.max() - sagittal_slice.min())
    coronal_slice = (coronal_slice - coronal_slice.min()) / (coronal_slice.max() - coronal_slice.min())

    return axial_slice, sagittal_slice, coronal_slice

In [12]:
class BrainMRIDataset(Dataset):
    def __init__(self, dataframe, root_dir, output_size):
        self.df = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.output_size = output_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subject_id = row['subject_id']
        age = row['subject_age']
        sex = 0 if row['subject_sex'] == 'm' else 1

        nii_path = os.path.join(self.root_dir, subject_id, 'anat', f'{subject_id}_T1w.nii.gz')
        axial, sagittal, coronal = load_middle_slices(nii_path)

        # Stack to tensor (3, H, W)
        image = np.stack([axial, sagittal, coronal], axis=0).astype(np.float32)
        image_tensor = torch.tensor(image)

        # Resize to (3, 128, 128)
        image_resized = resize(image_tensor, size=[self.output_size, self.output_size], interpolation=InterpolationMode.BILINEAR)

        while True:
            cf_idx = random.randint(0, len(self.df) - 1)
            if cf_idx != idx:
                break
        cf_row = self.df.iloc[cf_idx]
        cf_age = cf_row['subject_age']
        cf_sex = 0 if cf_row['subject_sex'] == 'm' else 1

        return {
            'image': image_resized,
            'age': torch.tensor(age, dtype=torch.float32),
            'sex': torch.tensor(sex, dtype=torch.long),
            'cf_age': torch.tensor(cf_age, dtype=torch.float32),
            'cf_sex': torch.tensor(cf_sex, dtype=torch.long),
            'subject_id': subject_id
        }


In [35]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = BrainMRIDataset(train_df, 'data', 128)
test_dataset = BrainMRIDataset(test_df, 'data', 128)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [36]:
print(f"Number of batches in train_loader: {len(train_loader)}")
print(f"Number of batches in test_loader: {len(test_loader)}")

Number of batches in train_loader: 1979
Number of batches in test_loader: 495


In [37]:
first_batch = next(iter(train_loader))
print("Image shape:", first_batch['image'].shape)  # (B, 3, H, W)
print("Age:", first_batch['age'])
print("Sex:", first_batch['sex'])
print("CF Age:", first_batch['cf_age'])
print("CF Sex:", first_batch['cf_sex'])
print("Subject IDs:", first_batch['subject_id'])

Image shape: torch.Size([2, 3, 128, 128])
Age: tensor([26., 71.])
Sex: tensor([1, 1])
CF Age: tensor([60., 72.])
CF Sex: tensor([1, 0])
Subject IDs: ['sub-BrainAge018990', 'sub-BrainAge020764']


In [44]:
# Age normalization
def normalize_age(age, min_age=18, max_age=97):
    return (age - min_age) / (max_age - min_age)

# Get example batch
batch = first_batch

x = batch['image']              # (B, 3, 130, 130)
cf_age = batch['cf_age']        # (B,)
cf_sex = batch['cf_sex']        # (B,)

# Normalize age
cf_age_norm = normalize_age(cf_age)  # (B,)

# Create condition vector (B, 2)
c_vec = torch.stack([cf_age_norm, cf_sex.float()], dim=1)

# Expand to (B, 2, 130, 130)
B, _, H, W = x.shape
c_map = c_vec.unsqueeze(-1).unsqueeze(-1).repeat(1, 1, H, W)

# Concatenate condition with input image => (B, 5, 130, 130)
x_input = torch.cat([x, c_map], dim=1)

print("Input to generator:", x_input.shape)

Input to generator: torch.Size([2, 5, 128, 128])


In [45]:
x_input.shape

torch.Size([2, 5, 128, 128])

In [46]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        
        # Encoder
        self.enc1 = nn.Conv2d(5, 64, kernel_size=4, stride=2, padding=1)  # (B, 5, 128, 128) -> (B, 64, 64, 64)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)  # (B, 64, 64, 64) -> (B, 128, 32, 32)
        self.enc3 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)  # (B, 128, 32, 32) -> (B, 256, 16, 16)
        
        # Decoder
        self.dec1 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)  # (B, 256, 16, 16) -> (B, 128, 32, 32)
        self.dec2 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)   # (B, 128, 32, 32) -> (B, 64, 64, 64)
        self.dec3 = nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1)     # (B, 64, 64, 64) -> (B, 3, 128, 128)

    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)

        # Decoder
        dec1 = self.dec1(enc3)
        dec2 = self.dec2(dec1)
        dec3 = self.dec3(dec2)

        return dec3

generator = Generator()
output = generator(x_input)
print(f"Output shape: {output.shape}")

Output shape: torch.Size([2, 3, 128, 128])


In [47]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        skip = x
        x = self.pool(x)
        return x, skip

class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()
        self.enc1 = UNetBlock(5, 16)      # Input: 5 channels (image + counterfactual condition)
        self.enc2 = UNetBlock(16, 32)

        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(32 + 16, 16)

        self.upconv2 = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(16 + 16, 16)

        self.final_conv = nn.Conv2d(16, 3, kernel_size=1)  # Output: 3-channel image

    def forward(self, x):
        # Encoding path
        x1, skip1 = self.enc1(x)   # (B, 16, 128, 128) → (B, 16, 64, 64)
        x2, skip2 = self.enc2(x1)  # (B, 32, 64, 64)  → (B, 32, 32, 32)

        # Decoding path
        x = self.upconv1(x2)                      # (B, 16, 64, 64)
        x = torch.cat([x, skip2], dim=1)          # (B, 48, 64, 64)
        x, _ = self.dec1(x)                       # (B, 16, 32, 32) → (B, 16, 32, 32)

        x = self.upconv2(x)                       # (B, 16, 64, 64) → (B, 16, 128, 128)
        x = torch.cat([x, skip1], dim=1)          # (B, 32, 128, 128)
        x, _ = self.dec2(x)                       # (B, 16, 128, 128)

        out = self.final_conv(x)                  # (B, 3, 128, 128)
        return out

generator = UNetGenerator()
output = generator(x_input)
print(f"Output shape: {output.shape}")


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 64 but got size 128 for tensor number 1 in the list.

In [32]:
# Định nghĩa Spatial Transformer Layer
class SpatialTransformer(nn.Module):
    def __init__(self):
        super(SpatialTransformer, self).__init__()

    def forward(self, x):
        # Áp dụng phép biến hình tại đây, sử dụng chức năng grid_sample của PyTorch
        theta = torch.zeros((x.size(0), 2, 3), device=x.device)  # Identity matrix
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        transformed_x = F.grid_sample(x, grid, align_corners=False)
        return transformed_x

# Cập nhật lại Generator để tích hợp Spatial Transformer
class UNetGeneratorWithSTL(UNetGenerator):
    def __init__(self):
        super(UNetGeneratorWithSTL, self).__init__()
        self.spatial_transformer = SpatialTransformer()

    def forward(self, x):
        # Encoder path
        enc1_out, skip1 = self.enc1(x)
        enc2_out, skip2 = self.enc2(enc1_out)
        
        # Decoder path với Spatial Transformer
        dec1_in = torch.cat([self.upconv1(enc2_out), skip2], dim=1)
        dec1_out, _ = self.dec1(dec1_in)
        
        # Áp dụng Spatial Transformer
        out = self.spatial_transformer(dec1_out)
        
        out = self.upconv2(out)
        return out

# Kiểm tra mô hình với Spatial Transformer
generator_stl = UNetGeneratorWithSTL()

# Chạy thử với mẫu input
output_stl = generator_stl(x_input)

# In kích thước đầu ra sau khi áp dụng Spatial Transformer
print(f"Output with Spatial Transformer shape: {output_stl.shape}")


Output with Spatial Transformer shape: torch.Size([2, 3, 64, 64])
